# Collaborative TMLE: which baseline variables belong in the assignment model?

This notebook estimates one average treatment effect with collaborative TMLE (C-TMLE). Each step
shows its code, its output, and what the output tells you.
[Collaborative TMLE](../technical-reference/collaborative-tmle.md) gives the candidate paths, the
selection loss, and the fold structure.

## The applied question

Clinical and operations staff approve three baseline variables for the navigation analysis. They
confirm that each variable is measured before assignment and that none is a collider. The review
cannot certify the causal role of each variable.

The analysis team asks which approved variables belong in the assignment model. An assignment model
that uses all three variables predicts assignment best. Predictive accuracy is the wrong criterion
for that model, and Step 8 shows why.

## What you will learn

| after this notebook you can | the step that shows it |
| --- | --- |
| write the protocol for a question with an approved baseline set | the protocol step |
| check that a method is available before you fit it | the design and identification step |
| fit C-TMLE with explicit learners, and read its selection path | Steps 6 and 7 |
| see what an instrument does to an assignment model | the failure mode |
| tell a selector that discriminates from one that selects nothing | the stress control |
| read the assessment of a selected working model | Steps 10 and 11 |


## Why this method

| your situation | what this method buys | what it costs |
| --- | --- | --- |
| an approved baseline set | an assignment model selected by cross-validated loss on the targeted outcome regression, so an instrument can be left out | one nuisance fit per candidate along the selection path |
| the outcome regression is already good | the empty assignment model is a legitimate candidate | selecting it is not evidence that the search discriminates |

The selector chooses a nuisance model inside the approved set. It does not discover a causal
adjustment set. The table below defines the terms this notebook uses most.

| term | plain meaning | canonical definition |
| --- | --- | --- |
| estimand | the number the question asks for, written before any model is chosen | [estimands](../user-guide/estimands.md) |
| nuisance | a model the estimate needs but the question does not ask about. Here, the outcome regression Q and the assignment model g | [point-treatment TMLE](../technical-reference/point-treatment-tmle.md) |
| targeting | a small update to Q, weighted by g, that removes first-order bias | [targeting and bounds](../user-guide/methods-learners.md#targeting-and-bounds) |
| influence curve | how much each row moves the estimate. Its variance gives the standard error | [inference](../technical-reference/inference.md) |
| positivity | every kind of patient has some chance of each arm | [diagnostics](../user-guide/results-assessment.md#diagnostics) |
| instrument | a variable that changes assignment and has no other path to the outcome | [collaborative TMLE](../technical-reference/collaborative-tmle.md#what-this-solves) |


## Step 1: set up

The setup imports the learners and prints the installed `cleverly` version. Every fit below passes
its learners, fold counts, and random seed explicitly, so a rerun reproduces the stored outputs.


In [1]:
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression

import cleverly

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 20)
print(f"cleverly {cleverly.__version__}")

cleverly 0.1.2


**What this output tells you.** The stored outputs in this notebook came from the version named
above. A different version can print different numbers.


## Step 2: the data

The data come from a synthetic law with a known answer. The generator is `make_instrument`. The code
renames its columns to the program's names, prints the first rows, and prints the true values of the
law.


In [2]:
from cleverly.datasets import make_instrument

frame, truth = make_instrument(n=2_000, seed=44)
frame = frame.rename(
    columns={
        "Y": "transition_score",
        "A": "transition_navigation",
        "W1": "baseline_readiness",
        "W2": "queue_lottery_draw",
        "W3": "social_support",
    }
)
lines = [
    f"rows and columns: {frame.shape}",
    frame.head().round(3).to_string(),
    "",
    "known values of the synthetic law:",
    *(f"  {key}: {truth[key]:.3f}" for key in ("ate", "att", "atc")),
]
print("\n".join(lines))

rows and columns: (2000, 5)
   transition_score  transition_navigation  baseline_readiness  queue_lottery_draw  social_support
0             3.472                    0.0               1.446               0.102           0.327
1             2.981                    1.0               1.136               0.824           0.579
2             2.604                    1.0              -0.379              -1.019           0.376
3             1.199                    1.0              -0.512               0.827           0.211
4             1.866                    1.0              -0.745               0.268           0.709

known values of the synthetic law:
  ate: 1.000
  att: 1.000
  atc: 1.000


**What this output tells you.** Each row is one discharge. `transition_navigation` is 1 for an
offer and 0 for usual support. The three baseline covariates are standardized (mean 0, SD 1), and
the score is in synthetic units.

The effect is constant in this law, so the population `ate`, `att`, and `atc` all equal 1.000. The
three columns have separate roles in the law.

| column | role in the law | what it is in the program |
| --- | --- | --- |
| `baseline_readiness` | confounder | navigators prioritize patients ready to engage, so higher readiness raises the chance of an offer and the transition score |
| `queue_lottery_draw` | instrument | an encounter-ID hash sets a queue draw. A higher draw strongly raises the chance of an offer and has no other path to the score |
| `social_support` | outcome predictor | it moves the transition score and does not move assignment |

The queue draw is an instrument only under three conditions. The program fixes the hash before
assignment, prevents staff overrides, and verifies that the draw changes no other service. The data
cannot establish this exclusion restriction.

A real program has no `truth`. Every comparison against it below is a teaching device.


## Step 3: association first

A confounder changes both who receives the offer and the outcome. The code compares the two arms
before any adjustment. It prints the mean score and the mean of each baseline covariate by arm.


In [3]:
covariates = ["baseline_readiness", "queue_lottery_draw", "social_support"]
by_arm = frame.groupby("transition_navigation")[["transition_score", *covariates]].mean()
print(by_arm.round(3))
print()
print(f"share offered navigation: {frame['transition_navigation'].mean():.3f}")
unadjusted = by_arm.loc[1.0, "transition_score"] - by_arm.loc[0.0, "transition_score"]
print(f"unadjusted difference in mean score: {unadjusted:.3f}")
print(f"population ATE:                      {truth['ate']:.3f}")

                       transition_score  baseline_readiness  queue_lottery_draw  social_support
transition_navigation                                                                          
0.0                               0.517              -0.288              -0.514          -0.035
1.0                               2.494               0.321               0.476           0.042

share offered navigation: 0.512
unadjusted difference in mean score: 1.977
population ATE:                      1.000


**What this output tells you.** The offered patients score 1.977 points higher on average, and the
true effect is 1.000. The arms differ before the offer. The mean `baseline_readiness` is 0.321
among offered patients and -0.288 among the others.

The mean `queue_lottery_draw` also differs between the arms, 0.476 against -0.514. That gap does
not bias the comparison, because the draw has no path to the score in this law. The readiness gap
does bias it. Neither the unadjusted comparison nor a model that predicts assignment can tell the
two gaps apart.


## Step 4: write the protocol

A `StudyProtocol` records the scientific design before any model runs. The record gets a
fingerprint, and every result fitted from it carries that fingerprint. This page starts from
`navigation_protocol()`, the protocol of the
[shared study design](index.md#the-shared-study-design). `dataclasses.replace` adds the approved
baseline review.

In [4]:
from dataclasses import replace

from cleverly.datasets import navigation_protocol

program = navigation_protocol()
protocol = replace(
    program,
    time_zero=(
        "Discharge-home order, after baseline measurement and the queue draw, "
        "and before the navigation offer"
    ),
    assumption_rationale=(
        "Clinical and operations review approved three baseline variables, each measured "
        "before assignment and none a collider",
        "The approved baseline variables cover the measured common causes of the offer "
        "and the score",
        "An encounter-ID hash fixes the queue draw before assignment, without staff overrides",
        *program.assumption_rationale[1:],
    ),
)
print("\n".join(protocol.summary_lines()))

causal study protocol: schema 1; 357de9af34828b51
target population: Adults with a discharge-home order at a participating hospital during the enrollment period
eligibility: ['Age 18 years or older', 'Discharge home ordered at a participating hospital']
time zero: Discharge-home order, after baseline measurement and the queue draw, and before the navigation offer
treatment strategies: ['Offer standard transition navigation', 'Provide usual discharge support']
treatment versions: ['Bedside transition plan and two scheduled navigator contacts within 30 days', 'No access to the transition-navigation offer']
outcome: Patient-reported transition score
horizon: 30 days after discharge
intercurrent-event handling: ['Use the transition score regardless of readmission', 'Analyze the offer regardless of completed contacts', 'The protocol scores death before day 30 as the worst transition score (composite strategy)']
interference unit: Individual patient
assumption rationale: ['Clinical and opera

**What this output tells you.** The first line gives the schema version and the fingerprint
`357de9af34828b51`. The other lines repeat each field. This page changes two fields of
`navigation_protocol()`, and [point-treatment TMLE](point-treatment-tmle.ipynb) reads the others.

| protocol field | what this page adds |
| --- | --- |
| time zero | the queue draw happens before the offer, so the draw is a baseline variable |
| assumption rationale | the approved baseline review, and the hash that fixes the queue draw before assignment |

Two design choices have no `StudyProtocol` field. The candidate variables for g are the adjustment
columns of the design in Step 5. The selection rule belongs to the method in Step 6. The exclusion
restriction for the queue draw has no field either, and the data cannot verify it.

## Step 5: design and identification

The design holds all three approved baseline columns. In this synthetic law, `baseline_readiness`
alone closes the common-cause path. C-TMLE selects terms for the assignment nuisance, and it does
not revise that identification decision.

The code identifies the ATE and prints its summary. It then lists the methods available for the
ATE, and asks whether C-TMLE is available for the ATT.


In [5]:
from cleverly import ATE, ATT, CausalStudy, PointTreatment

study = CausalStudy(
    frame,
    design=PointTreatment(
        outcome="transition_score",
        treatment="transition_navigation",
        adjustment=("baseline_readiness", "queue_lottery_draw", "social_support"),
    ),
    protocol=protocol,
)
effect = study.identify(ATE(reference=0))
att_methods = {
    method.name: method for method in study.identify(ATT(reference=0)).available_methods()
}
att_collaborative = att_methods["collaborative_tmle"]
print(effect.summary())
print()
print("methods for the ATE:")
for method in effect.available_methods():
    print(f"  {method.name}: available={method.available} {method.reason or ''}")
print()
print(f"collaborative_tmle for the ATT: available={att_collaborative.available}")
print(f"  reason: {att_collaborative.reason}")

average treatment effect, E[Y^a] - E[Y^reference]
identified by explicit-adjustment: E_W[E(transition_score | transition_navigation=a, W)] - E_W[E(transition_score | transition_navigation=0, W)] for a in [1]
adjustment/history: ['baseline_readiness', 'queue_lottery_draw', 'social_support']
required nuisances: ['outcome_regression', 'treatment_mechanism']
assumptions:
  - consistency: Y = Y^a when A = a
  - no interference: one unit's potential outcome does not depend on other units' treatment assignments
  - no unmeasured confounding: Y^a is independent of A given W
  - positivity: P(transition_navigation = a | W) > 0 almost surely for every supported treatment level a in [0, 1]
causal study protocol: schema 1; 357de9af34828b51
target population: Adults with a discharge-home order at a participating hospital during the enrollment period
eligibility: ['Age 18 years or older', 'Discharge home ordered at a participating hospital']
time zero: Discharge-home order, after baseline measuremen

**What this output tells you.** The first lines show the ATE and its observed-data formula. The
formula averages the outcome regression over all patients, once with the offer and once without.
The required nuisances are the outcome regression Q and the treatment mechanism g. The summary then
lists four assumptions and repeats the stored protocol.

The four assumptions are those of
[point-treatment TMLE](point-treatment-tmle.ipynb#step-5-design-and-identification), which reads
each one for the program. Here no unmeasured confounding means that the three approved variables
block every common cause of the offer and the score. The data cannot check it.

The method list shows `collaborative_tmle` as available for the ATE. For the ATT, the catalog
refuses it with the reason `no collaborative score is evidenced for this functional`. C-TMLE covers
the point-treatment `ate`, `ey`, `ey1`, `ey0`, `rr`, and `or` targets only. The
[technical entry](../technical-reference/collaborative-tmle.md#what-this-solves) states that scope.


## Step 6: estimate the ATE with C-TMLE

The configuration is written out in full. Linear learners are enough here, because the outcome
mean of this law is linear in the offer and the covariates.

| setting | value | what it does |
| --- | --- | --- |
| `outcome_learner` | linear regression | fits Q, the expected score given the offer and the covariates |
| `treatment_learner` | logistic regression | fits each candidate g on the covariates that candidate uses |
| `CrossFitting(n_folds=3)` | three folds | predicts each row's nuisances from models that did not see that row |
| `strategy="greedy"` | greedy path | at each stage, adds to g the covariate whose targeted outcome regression has the smallest penalized loss |
| `selection_folds=3` | three selection folds | cross-validates the loss that chooses the stopping point on the path |
| `selection_inner_folds=2` | two inner folds | limits the extra fits inside each selection fold |
| `Runtime(random_state=44, n_jobs=1)` | fixed seed, one process | makes the fit reproducible |

Cross-fitting means each row's nuisance prediction comes from models fit without that row. The
[CV-TMLE entry](../technical-reference/cv-tmle.md) defines it. The selection folds are a separate
layer, and the
[technical entry](../technical-reference/collaborative-tmle.md#the-algorithm-as-implemented)
describes both. The plain TMLE fit in Step 8 uses the same learners and folds. Only the choice of
assignment model differs.


In [6]:
from cleverly import CollaborativeTMLEMethod, CrossFitting, ModelSpec, Runtime, TMLEMethod

models = ModelSpec(
    outcome_learner=LinearRegression(),
    treatment_learner=LogisticRegression(max_iter=1000),
)
folds = CrossFitting(n_folds=3)
runtime = Runtime(random_state=44, n_jobs=1)

collaborative = effect.estimate(
    method=CollaborativeTMLEMethod(
        models=models,
        cross_fitting=folds,
        runtime=runtime,
        strategy="greedy",
        selection_folds=3,
        selection_inner_folds=2,
    )
)
point = collaborative["ate"]
print(collaborative.summary())
print()
print(f"estimate:        {point.psi:.3f}")
print(f"standard error:  {point.std_error:.4f}")
print(f"95% CI:          ({point.ci[0]:.3f}, {point.ci[1]:.3f})")
print(f"population ATE:  {truth['ate']:.3f}")

Targeted maximum likelihood estimation
n = 2000; covariates = 3; P(A=1) = 0.512
causal estimand: average treatment effect, E[Y^a] - E[Y^reference]
identification: explicit-adjustment; E_W[E(transition_score | transition_navigation=a, W)] - E_W[E(transition_score | transition_navigation=0, W)] for a in [1]
required nuisances: outcome_regression, treatment_mechanism
identification assumptions: consistency: Y = Y^a when A = a; no interference: one unit's potential outcome does not depend on other units' treatment assignments; no unmeasured confounding: Y^a is independent of A given W; positivity: P(transition_navigation = a | W) > 0 almost surely for every supported treatment level a in [0, 1]
causal study protocol: schema 1; 357de9af34828b51
target population: Adults with a discharge-home order at a participating hospital during the enrollment period
eligibility: ['Age 18 years or older', 'Discharge home ordered at a participating hospital']
time zero: Discharge-home order, after baselin

**What this output tells you.** The summary repeats the estimand, the identification, and the
protocol with its fingerprint. It names the construction as stacked CV-TMLE, with nuisances
cross-fitted over 3 folds and pooled targeting. The summary does not name the selector. Step 7
reads the selection from the nuisance report.

The estimate is 0.955 with a standard error of 0.0441. The 95% interval is (0.868, 1.041), and it
contains the true ATE of 1.000. That is one draw, not a coverage result. The interval treats the
selected assignment model as fixed in advance, and the trust section explains that limit.


## Step 7: read the selection path

The nuisance report retains the selection path. It lists the candidate assignment models in order
and marks the one that the cross-validated loss chose.


In [7]:
selection = collaborative.diagnostics.nuisance_models().selection
print(selection.summary())
print()
print(f"candidate path: {selection.path}")
print(f"selected covariates: {selection.selected_covariates}")

Collaborative TMLE selection
strategy = greedy; preorder = n/a; target = ate (ate); criterion = cross-validated penalized squared-error loss

k  covariates in g                                         steps  risk     cv risk     
-  ------------------------------------------------------  -----  -------  -------  ---
0  (intercept)                                             1      6.59305  6.59286  <--
1  social_support                                          1      6.59305  6.59304     
2  social_support, baseline_readiness                      2      6.59442  6.59616     
3  social_support, baseline_readiness, queue_lottery_draw  3      6.60011  6.60551     

selected g: (intercept only)
left out: baseline_readiness, queue_lottery_draw, social_support. The selected candidate minimized targeted cross-validated penalized squared-error loss; this criterion does not determine why a covariate was left out

candidate path: ((), ('social_support',), ('social_support', 'baseline_readiness')

**What this output tells you.** The table lists four candidates for g. The greedy path starts
with the intercept only. It then adds `social_support`, `baseline_readiness`, and
`queue_lottery_draw`, in that order. The marker `<--` is on row 0, so the selected g is intercept
only.

The `cv risk` column decides the choice. The intercept-only candidate has the smallest value,
6.59286. The candidate with all three variables has the largest value, 6.60551. The footer names
the loss that chose the candidate, and it says that this criterion does not determine why a
covariate was left out.

The linear outcome regression is correctly specified for this law. The empty g is then a legitimate
choice. Step 9 explains why that choice alone does not test the selector.


## Step 8: the failure mode, an instrument in the assignment model

A plain TMLE fit puts all three approved variables into g. The code fits it with the same learners
and folds, and it compares the fitted propensity tails of both fits. The support report describes
how far the fitted propensities reach toward 0 and 1. The code then prints each ATE estimate.

An instrument in g pushes propensity scores toward 0 and 1 without removing confounding, so
precision falls. The clever covariate divides by the fitted propensity, so an extreme propensity
makes it large. [Point-treatment TMLE](../technical-reference/point-treatment-tmle.md) defines it.

The precision argument assumes exchangeability. If an unmeasured common cause remains, a strong
instrument can also amplify residual bias. C-TMLE does not turn the queue draw into a design-based
instrument estimator.


In [8]:
plain = effect.estimate(method=TMLEMethod(models=models, cross_fitting=folds, runtime=runtime))


def tails(result):
    support = result.diagnostics.support()
    return {
        "share of g below 0.1": support.tail_mass[0.1]["below"],
        "share of g above 0.9": support.tail_mass[0.1]["above"],
        "truncated fraction": support.truncated["fraction"],
        "treated ESS / n": support.effective_sample_size["treated"]["ratio"],
        "control ESS / n": support.effective_sample_size["control"]["ratio"],
        "max |clever covariate|": support.clever_covariate_max["mean"],
    }


tail_table = pd.DataFrame({"plain TMLE": tails(plain), "collaborative TMLE": tails(collaborative)})
print(tail_table.round(4))
print()


def show(label, result):
    point = result["ate"]
    low, high = point.ci
    print(
        f"{label:22s} psi={point.psi:6.3f}  se={point.std_error:6.4f}  CI=({low:.3f}, {high:.3f})"
    )


show("plain TMLE", plain)
show("collaborative TMLE", collaborative)
print(f"population ATE: {truth['ate']:.3f}")

                        plain TMLE  collaborative TMLE
share of g below 0.1        0.1015              0.0000
share of g above 0.9        0.1140              0.0000
truncated fraction          0.0160              0.0000
treated ESS / n             0.4704              1.0000
control ESS / n             0.4422              1.0000
max |clever covariate|     43.7888              2.0508

plain TMLE             psi= 0.882  se=0.0637  CI=(0.757, 1.007)
collaborative TMLE     psi= 0.955  se=0.0441  CI=(0.868, 1.041)
population ATE: 1.000


**What this output tells you.** In the plain fit, a share of 0.1015 of the fitted propensities
lies below 0.1. A share of 0.1140 lies above 0.9. Its largest clever covariate is 43.7888. Its
effective-sample ratios are 0.4704 in the treated arm and 0.4422 in the control arm. That is the
queue lottery at work. High draws push propensities toward one, and low draws push them toward
zero.

The instrument removes no confounding in exchange, because the draw has no path to the score in
this law. The collaborative fit has no rows in either tail and an effective-sample ratio of 1.0000
in each arm.

The plain standard error is 0.0637, and the collaborative standard error is 0.0441. Both intervals
contain the true value of 1.000 on this draw. The next step explains why this comparison does not
show that the selector works.


## Step 9: a control that discriminates

With a correctly specified outcome model, Step 7 showed the selector choose the empty assignment
model. That choice can minimize the targeted cross-validated loss. It does not show that the search
can tell a confounder from an instrument. A selector that always chose the empty model would give
the same result.

Use a deliberate stress control where selecting nothing is wrong. The code reduces the outcome model
to a constant with `DummyRegressor`, so the assignment model must carry the adjustment. This tests
the selector. It is not a recommended production outcome model.


In [9]:
weak_models = ModelSpec(
    outcome_learner=DummyRegressor(),
    treatment_learner=LogisticRegression(max_iter=1000),
)
weak_plain = effect.estimate(
    method=TMLEMethod(models=weak_models, cross_fitting=folds, runtime=runtime)
)
weak_collaborative = effect.estimate(
    method=CollaborativeTMLEMethod(
        models=weak_models,
        cross_fitting=folds,
        runtime=runtime,
        strategy="greedy",
        selection_folds=3,
        selection_inner_folds=2,
    )
)
weak_selection = weak_collaborative.diagnostics.nuisance_models().selection
ratio = weak_collaborative["ate"].std_error / weak_plain["ate"].std_error
show("constant Q, plain", weak_plain)
show("constant Q, C-TMLE", weak_collaborative)
print(f"standard-error ratio, C-TMLE / plain: {ratio:.3f}")
print(f"population ATE: {truth['ate']:.3f}")
print()
print(weak_selection.summary())

constant Q, plain      psi= 1.057  se=0.2386  CI=(0.590, 1.525)
constant Q, C-TMLE     psi= 1.014  se=0.0927  CI=(0.832, 1.196)
standard-error ratio, C-TMLE / plain: 0.389
population ATE: 1.000

Collaborative TMLE selection
strategy = greedy; preorder = n/a; target = ate (ate); criterion = cross-validated penalized squared-error loss

k  covariates in g                                         steps  risk     cv risk     
-  ------------------------------------------------------  -----  -------  -------  ---
0  (intercept)                                             1      26.3478  26.3541     
1  baseline_readiness                                      2      25.0381  25.0451     
2  baseline_readiness, social_support                      2      24.9424  24.948   <--
3  baseline_readiness, social_support, queue_lottery_draw  3      24.9901  25.108      

selected g: baseline_readiness, social_support
left out: queue_lottery_draw. The selected candidate minimized targeted cross-validated

**What this output tells you.** With a constant Q, the selected g is
`baseline_readiness, social_support`. The footer lists `queue_lottery_draw` as left out. On this
draw the selector keeps the confounder and leaves the instrument out.

The path table shows the change in the loss. Adding `baseline_readiness` lowers the `cv risk` from
26.3541 to 25.0451. Adding `queue_lottery_draw` to the other two raises it from 24.948 to 25.108.

The C-TMLE standard error is 0.0927, and the plain standard error is 0.2386. The ratio is 0.389, so
the selected fit's standard error is less than half that of the plain fit. Both intervals contain
1.000.

In this known law, the search keeps the variable that the constant outcome model needs. It drops the
pure assignment predictor. The selection result does not prove that either variable has its
declared causal role.


## Step 10: diagnostics, what the fit can check

Start with the combined assessment of the Step 6 fit. It presents validation, diagnostics, and
sensitivity together. The code then prints the support report and the nuisance report, with the role
the nuisance report gives to g.


In [10]:
assessment = collaborative.assess()
support_summary = assessment.report("support").summary()
nuisance = assessment.report("nuisance_models")
print(assessment.summary())
print(f"needs attention: {tuple(item.name for item in assessment.attention)}")
print()
print(support_summary)
print()
print(f"treatment role: {nuisance.treatment_role}")
print(nuisance.summary())
print()
print(f"propensity AUC: {nuisance['propensity'].metrics['auc']:.3f}")
print(f"selected covariates: {nuisance.selection.selected_covariates}")

Returned results
----------------
surface      operation            result                                                                                                                                                                                    
-----------  -------------------  ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
validation   support              maximum truncated fraction 0.0%; minimum effective-sample-size ratio 100.0%; group load: mean:h0 976.0/2000 Kish-equivalent mask rows (48.8%; 48.8% all; draw 01 of 01); not estimator ESS
validation   nuisance_models      2 nuisance model report(s) are available; C-TMLE greedy selected candidate 1 of 4 for ate                                                                                                 
sensitivity  omitted_confounding  at the default strengths; cf_y=0.03, cf_d=0.03, 

**What this output tells you.** Read the four parts in order.

| output part | what it shows on this draw |
| --- | --- |
| `Checks` and `needs attention` | the score-equation check passed, and nothing needs attention |
| `Returned results` | `support` and `nuisance_models` are `completed`. A `completed` row means the calculation ran. It is not a pass |
| positivity and overlap | every fitted propensity lies between 0.5116 and 0.5124, and no row is truncated |
| nuisance model diagnostics | the treatment role is `collaborative_working_model`, and the propensity AUC is 0.499 |

The selected model is intercept only, so its AUC is about 0.5 by construction. Its calibration
slope of -1.9985 is not meaningful for a constant model. These metrics describe the working
denominator, not assignment given the complete adjustment set.
[Nuisance model quality](../technical-reference/validation-methods.md#nuisance-model-quality) lists
the two claims that this role drops.

The support report describes the selected denominator only. Compare it with the plain fit in Step 8
to see the tails that selection removed. Neither report assigns a causal role to an omitted
variable.


## Step 11: sensitivity, what the fit cannot check

Sensitivity analysis asks how strong an unmeasured confounder would need to be to change the
conclusion. The [results guide](../user-guide/results-assessment.md#sensitivity-analysis) introduces
it. Sensitivity analysis cannot tell whether the selector chose a useful assignment model. The code
prints the omitted-variable elements and the robustness value for both fits, so you can see what
the selected g changes.


In [11]:
sensitivity_rows = {}
for label, fitted, fitted_assessment in (
    ("plain TMLE", plain, plain.assess()),
    ("collaborative TMLE", collaborative, assessment),
):
    elements = fitted_assessment.report("elements")
    robustness = fitted_assessment.report("robustness_value")
    sensitivity_rows[label] = {
        "estimate": fitted["ate"].psi,
        "sigma2": elements.sigma2,
        "nu2": elements.nu2,
        "robustness value": robustness["rv"],
        "confidence-limit value": robustness["rva"],
    }
print(pd.DataFrame(sensitivity_rows).round(3))

                        plain TMLE  collaborative TMLE
estimate                     0.882               0.955
sigma2                       0.972               0.974
nu2                         11.765               4.002
robustness value             0.229               0.381
confidence-limit value       0.205               0.357


**What this output tells you.** The plain fit has a robustness value of 0.229, and the
collaborative fit has 0.381. The residual variance `sigma2` is almost the same in both fits, 0.972
and 0.974. The difference comes from `nu2`, which is 11.765 for the plain fit and 4.002 for the
collaborative fit.

The
[omitted-variable bounds](../technical-reference/validation-methods.md#omitted-variable-bounds-robustness-value-benchmark-and-contours)
define `nu2` from the Riesz representer. On a collaborative fit, the code builds that representer
from the selected working g. Here that g is intercept only, so `nu2` is small and the robustness
value is large.

Do not read the larger value as a more robust conclusion. The reference gives no derivation of the
bound for a selected working mechanism. Report both values, and name the g that each value reads.
Neither value shows that no unmeasured confounder exists.


## How far to trust this

Two limits belong in every report of a collaborative fit.

**Post-selection coverage has not been established.** The data chose the candidate model. The
reported Wald interval treats that model as if it had been fixed in advance.

**The influence curve can omit a first-order term.** This happens when the selected model is not
consistent for the true assignment mechanism. The empty model in Step 7 is such a case, so its
reported standard error can be too small. The
[technical entry](../technical-reference/collaborative-tmle.md#validation-issues-special-to-this-method)
records both limits. No diagnostic on the fit repairs them.

The
[selector-based point-treatment C-TMLE study](../technical-reference/method-evidence/selector-based-point-treatment-c-tmle.md)
validates the greedy selector against R `ctmle` with logistic GLMs. Its law has a binary outcome.
No registered study in that entry covers the continuous score used here.

| layer | establishes | does not establish |
| --- | --- | --- |
| the combined assessment | which cached checks need attention and which costly operations did not run | selection uncertainty or the causal role of a candidate variable |
| the support report | how far the selected denominator reaches into the tails | that the selected model is the right one |
| the nuisance report | selected-model metrics, model role, and the retained selection | whether low AUC means limited confounding after collaborative selection |
| the retained selection path | which candidates the search considered and selected | calibrated inference for the selected candidate |
| the sensitivity analysis | a robustness value computed from the selected g | a bound that accounts for the instrument or the omitted covariates |
| the registered study | the greedy selector recovers known truths on its binary-outcome law | that your identification assumptions hold on your data |


## Where to go next

Collaborative TMLE addresses *selection*. If your worry is the *inference* instead, because you
expect one nuisance to be inconsistent however you choose it, read [DR-TMLE](dr-tmle.ipynb). If the
adjustment set is small and you would include all of it, the plain
[point-treatment TMLE](point-treatment-tmle.ipynb) is the right entry.

The library refuses longitudinal and incremental-target C-TMLE. The
[technical entry](../technical-reference/collaborative-tmle.md#variations) gives the reasons. The
[examples index](index.md#the-program) lists every tutorial in the program.
